### AV平仓错误：UnboundLocalError: cannot access local variable 'volume' where it is not associated with a value

```log
2025-06-20 10:03:40.606 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-20 10:03:40.624 | INFO | SimpleStrategy | combined: AV走势特别平仓遇到错误 (sc2508P405.INE) Traceback (most recent call last):
  File "c:\vnpy\udt\udt\strategies\combined.py", line 1101, in close_positions_for_AV
    combined_volume: int = round((1 - self.combined_volumes(data['vt_symbol'], Direction.LONG) / self.combined_volumes(data['vt_symbol'], Direction.SHORT)) * volume)
                                                                                                                                                              ^^^^^^
UnboundLocalError: cannot access local variable 'volume' where it is not associated with a value
```

直接原因是 volume 没有赋值。根本原因是没有找到持仓。

暂时的解决办法：判断 volume 未赋值则不进行 AV 平仓。

### AV平仓错误：ZeroDivisionError: float division by zero


```log
2025-06-20 10:04:58.523 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-20 10:04:58.546 | INFO | SimpleStrategy | combined: AV走势特别平仓遇到错误 (sc2509P415.INE) Traceback (most recent call last):
  File "c:\vnpy\udt\udt\strategies\combined.py", line 1101, in close_positions_for_AV
    combined_volume: int = round((1 - self.combined_volumes(data['vt_symbol'], Direction.LONG) / self.combined_volumes(data['vt_symbol'], Direction.SHORT)) * volume)
                                      ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
ZeroDivisionError: float division by zero
```

### 止盈平仓时：CTP：平仓量超过持仓量

中途随便截取的日志, 当时确认了一下平仓单其实已经发出去了, 但不知道为什么还是在止盈平仓

```log
2025-06-20 10:01:09.772 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-20 10:01:09.791 | INFO | SimpleStrategy | combined: 止盈 请求平仓53 合约=MA508C2850.CZCE 方向=Direction.LONG 手数=1 @0.5
2025-06-20 10:01:09.791 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
```

OK, 问题稍微有点头绪了😄

晚上9点整截取了第一次行情和第二次行情的日志, 这次似乎更加清晰一点.

可以看到第一次止盈平仓没有问题, 第二次止盈平仓就出现了CTP报错.

查看无限易, 发现第一次止盈平仓单其实已经发出去了.

我怀疑是持仓数据没有及时更新的原因, 导致策略依然以过时的持仓数据做出判断来执行操作.

需要仔细查看 vnpy 的底层实现, 重点关注 vnpy_gateway.

```log
2025-06-20 21:00:07.436 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-20 21:00:18.407 | INFO | SimpleStrategy | combined: 止盈 请求平仓0 合约=CF509P11000.CZCE 方向=Direction.LONG 手数=1 @3.0
2025-06-20 21:00:18.407 | INFO | SimpleStrategy | combined: 止盈 请求平仓1 合约=CF509C15800.CZCE 方向=Direction.LONG 手数=1 @3.0
2025-06-20 21:00:18.418 | INFO | SimpleStrategy | combined: 止盈 请求平仓2 合约=MA508C2850.CZCE 方向=Direction.LONG 手数=2 @0.5
2025-06-20 21:00:18.420 | INFO | SimpleStrategy | combined: 止盈 请求平仓3 合约=OI509P7300.CZCE 方向=Direction.LONG 手数=1 @1.5
2025-06-20 21:00:18.425 | INFO | SimpleStrategy | combined: 止盈 请求平仓4 合约=PF508P6000.CZCE 方向=Direction.LONG 手数=1 @0.5
2025-06-20 21:00:18.427 | INFO | SimpleStrategy | combined: 止盈 请求平仓5 合约=PF508P5900.CZCE 方向=Direction.LONG 手数=4 @0.5
2025-06-20 21:00:18.432 | INFO | SimpleStrategy | combined: 止盈 请求平仓6 合约=PR508P5500.CZCE 方向=Direction.LONG 手数=2 @0.5
2025-06-20 21:00:18.433 | INFO | SimpleStrategy | combined: 止盈 请求平仓7 合约=PR508P5400.CZCE 方向=Direction.LONG 手数=2 @0.5
2025-06-20 21:00:18.434 | INFO | SimpleStrategy | combined: 止盈 请求平仓8 合约=RM508P2350.CZCE 方向=Direction.LONG 手数=1 @0.5
2025-06-20 21:00:18.460 | INFO | SimpleStrategy | combined: 止盈 请求平仓9 合约=TA509P3800.CZCE 方向=Direction.LONG 手数=11 @1.5
2025-06-20 21:00:18.462 | INFO | SimpleStrategy | combined: 止盈 请求平仓10 合约=TA508P4250.CZCE 方向=Direction.LONG 手数=1 @0.5
2025-06-20 21:00:18.471 | INFO | SimpleStrategy | combined: 止盈 请求平仓11 合约=ao2508P2450.SHFE 方向=Direction.LONG 手数=4 @1.5
2025-06-20 21:00:18.481 | INFO | SimpleStrategy | combined: 止盈 请求平仓12 合约=eb2508-C-9800.DCE 方向=Direction.LONG 手数=2 @0.5
2025-06-20 21:00:18.487 | INFO | SimpleStrategy | combined: 止盈 请求平仓13 合约=ni2508C148000.SHFE 方向=Direction.LONG 手数=1 @6.0
2025-06-20 21:00:18.492 | INFO | SimpleStrategy | combined: 止盈 请求平仓14 合约=pg2508-P-3500.DCE 方向=Direction.LONG 手数=26 @0.2
2025-06-20 21:00:23.385 | INFO | SimpleStrategy | combined: 止盈 请求平仓15 合约=sc2508P395.INE 方向=Direction.LONG 手数=7 @0.05
2025-06-20 21:00:23.391 | INFO | SimpleStrategy | combined: 止盈 请求平仓16 合约=v2509-C-7000.DCE 方向=Direction.LONG 手数=43 @1.5
2025-06-20 21:00:23.391 | INFO | SimpleStrategy | combined: 止盈 请求平仓17 合约=zn2508P19000.SHFE 方向=Direction.LONG 手数=2 @3.0
2025-06-20 21:00:23.396 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-20 21:00:28.358 | INFO | SimpleStrategy | combined: 止盈 请求平仓18 合约=CF509P11000.CZCE 方向=Direction.LONG 手数=1 @3.0
2025-06-20 21:00:28.358 | INFO | SimpleStrategy | combined: 止盈 请求平仓19 合约=CF509C15800.CZCE 方向=Direction.LONG 手数=1 @3.0
2025-06-20 21:00:28.359 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.359 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.359 | INFO | SimpleStrategy | combined: 止盈 请求平仓20 合约=MA508C2850.CZCE 方向=Direction.LONG 手数=2 @0.5
2025-06-20 21:00:28.359 | INFO | SimpleStrategy | combined: 止盈 请求平仓21 合约=OI509P7300.CZCE 方向=Direction.LONG 手数=1 @1.5
2025-06-20 21:00:28.360 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.360 | INFO | SimpleStrategy | combined: 止盈 请求平仓22 合约=PF508P6000.CZCE 方向=Direction.LONG 手数=1 @0.5
2025-06-20 21:00:28.360 | INFO | SimpleStrategy | combined: 止盈 请求平仓23 合约=PF508P5900.CZCE 方向=Direction.LONG 手数=4 @0.5
2025-06-20 21:00:28.360 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.360 | INFO | SimpleStrategy | combined: 止盈 请求平仓24 合约=PR508P5500.CZCE 方向=Direction.LONG 手数=2 @0.5
2025-06-20 21:00:28.361 | INFO | SimpleStrategy | combined: 止盈 请求平仓25 合约=PR508P5400.CZCE 方向=Direction.LONG 手数=2 @0.5
2025-06-20 21:00:28.361 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.361 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.362 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.362 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.362 | INFO | SimpleStrategy | combined: 止盈 请求平仓26 合约=RM508P2350.CZCE 方向=Direction.LONG 手数=1 @0.5
2025-06-20 21:00:28.364 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.365 | INFO | SimpleStrategy | combined: 止盈 请求平仓27 合约=TA509P3800.CZCE 方向=Direction.LONG 手数=11 @1.5
2025-06-20 21:00:28.365 | INFO | SimpleStrategy | combined: 止盈 请求平仓28 合约=TA508P4250.CZCE 方向=Direction.LONG 手数=1 @0.5
2025-06-20 21:00:28.365 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.365 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.366 | INFO | SimpleStrategy | combined: 止盈 请求平仓29 合约=ao2508P2450.SHFE 方向=Direction.LONG 手数=4 @1.5
2025-06-20 21:00:28.369 | INFO | SimpleStrategy | combined: 止盈 请求平仓30 合约=eb2508-C-9800.DCE 方向=Direction.LONG 手数=2 @0.5
2025-06-20 21:00:28.369 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.370 | INFO | SimpleStrategy | combined: 止盈 请求平仓31 合约=ni2508C148000.SHFE 方向=Direction.LONG 手数=1 @6.0
2025-06-20 21:00:28.370 | INFO | SimpleStrategy | combined: 止盈 请求平仓32 合约=pg2508-P-3500.DCE 方向=Direction.LONG 手数=26 @0.2
2025-06-20 21:00:28.371 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.372 | INFO | SimpleStrategy | combined: 止盈 请求平仓33 合约=sc2508P395.INE 方向=Direction.LONG 手数=7 @0.05
2025-06-20 21:00:28.373 | INFO | SimpleStrategy | combined: 止盈 请求平仓34 合约=v2509-C-7000.DCE 方向=Direction.LONG 手数=43 @1.5
2025-06-20 21:00:28.373 | INFO | CTP | 交易委托失败，代码：30，信息：CTP:平仓量超过持仓量
2025-06-20 21:00:28.373 | INFO | SimpleStrategy | combined: 止盈 请求平仓35 合约=zn2508P19000.SHFE 方向=Direction.LONG 手数=2 @3.0
```

### 掉线重连

看似没问题, 具体请看下列时间戳的日志:
- 2025-06-21 01:50:01.150 开始掉线.
- 2025-06-21 01:50:12.359 恢复连接.
- 2025-06-21 01:50:12.707 重新接收订单信息.
- 2025-06-21 01:50:16.311 行情服务器登录成功
- 2025-06-21 01:50:20.186 重新收到新的行情.


```log
2025-06-21 00:53:44.705 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 00:54:48.607 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 00:55:39.231 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 00:56:34.724 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 00:57:26.716 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 00:58:04.391 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 00:58:49.402 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 00:59:35.490 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:00:08.928 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:01:06.711 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:02:19.703 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:03:39.745 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:05:35.204 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:07:47.254 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:09:12.653 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:11:37.634 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:13:30.152 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:15:46.125 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:17:29.679 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:19:14.202 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:20:58.621 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:22:23.798 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:23:42.262 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:25:33.637 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:27:26.880 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:29:25.905 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:31:36.646 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:34:19.601 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:36:27.616 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:39:48.221 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:42:33.037 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:44:50.022 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:47:08.994 | INFO | SimpleStrategy | combined: 行情数据更新
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD36D623A8][-2130706431][ 4097]
2025-06-21 01:50:01.150 | INFO | CTP | 交易服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD36D617D8][-2130706431][ 4097]
2025-06-21 01:50:05.184 | INFO | CTP | 行情服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C687908][-1622605822][ 4097]
2025-06-21 01:50:06.166 | INFO | CTP | 交易服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C687588][-1622343678][ 4097]
2025-06-21 01:50:10.204 | INFO | CTP | 行情服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C687748][-1622278141][ 4097]
2025-06-21 01:50:11.174 | INFO | CTP | 交易服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C686248][-1621950460][ 4097]
2025-06-21 01:50:12.255 | INFO | CTP | 交易服务器连接断开，原因 4097
2025-06-21 01:50:12.359 | INFO | CTP | 交易服务器连接成功
2025-06-21 01:50:12.548 | INFO | CTP | 交易服务器授权验证成功
2025-06-21 01:50:12.591 | INFO | CTP | 交易服务器登录成功
2025-06-21 01:50:12.707 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=ni2507C138000.SHFE, 编号=       46649, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.CLOSE, 价格=16.0, 数量=3, Memo=
2025-06-21 01:50:12.721 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=sc2508P410.INE, 编号=       46648, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.CLOSE, 价格=0.5, 数量=3, Memo=
2025-06-21 01:50:12.784 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=CF509C15600.CZCE, 编号=2025062300038926, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.CLOSE, 价格=15.0, 数量=1, Memo=
2025-06-21 01:50:12.784 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=CF509P11000.CZCE, 编号=2025062300152298, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=3.0, 数量=1, Memo=0
2025-06-21 01:50:12.785 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=CF509C15800.CZCE, 编号=2025062300152299, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=3.0, 数量=1, Memo=1
2025-06-21 01:50:12.785 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=MA508C2850.CZCE, 编号=2025062300152504, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.5, 数量=2, Memo=2
2025-06-21 01:50:12.785 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=OI509P7300.CZCE, 编号=2025062300152550, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=1.5, 数量=1, Memo=3
2025-06-21 01:50:12.785 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=PF508P6000.CZCE, 编号=2025062300153070, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.5, 数量=1, Memo=4
2025-06-21 01:50:12.786 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=PF508P5900.CZCE, 编号=2025062300153071, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.5, 数量=4, Memo=5
2025-06-21 01:50:12.786 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=PR508P5500.CZCE, 编号=2025062300153387, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.5, 数量=2, Memo=6
2025-06-21 01:50:12.786 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=PR508P5400.CZCE, 编号=2025062300153388, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.5, 数量=2, Memo=7
2025-06-21 01:50:12.786 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=RM508P2350.CZCE, 编号=2025062300153807, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.5, 数量=1, Memo=8
2025-06-21 01:50:12.787 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=TA508P4250.CZCE, 编号=2025062300154955, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.5, 数量=1, Memo=10
2025-06-21 01:50:12.787 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=TA509P3800.CZCE, 编号=2025062300154956, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=1.5, 数量=11, Memo=9
2025-06-21 01:50:12.792 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=ao2508P2450.SHFE, 编号=      219008, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSEYESTERDAY, 价格=1.5,  数量=4, Memo=11
2025-06-21 01:50:12.792 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=eb2508-C-9800.DCE, 编号=   100173542, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.5, 数量=2, Memo=12
2025-06-21 01:50:12.792 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=ni2508C148000.SHFE, 编号=      223459, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSEYESTERDAY, 价格=6.0, 数量=1, Memo=13
2025-06-21 01:50:12.884 | INFO | CTP | 结算信息确认成功
2025-06-21 01:50:13.011 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-P-3500.DCE, 编号=   100174820, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.2, 数量=26, Memo=14
2025-06-21 01:50:13.097 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=sc2508P395.INE, 编号=      227171, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSEYESTERDAY, 价格=0.05, 数量=7, Memo=15
2025-06-21 01:50:13.261 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=zn2508P19000.SHFE, 编号=      227653, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSEYESTERDAY, 价格=3.0, 数量=2, Memo=17
2025-06-21 01:50:13.267 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=v2509-C-7000.DCE, 编号=   100176110, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=1.5, 数量=43, Memo=16
2025-06-21 01:50:13.361 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   102477206, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=17.6, 数量=1, Memo=90
2025-06-21 01:50:13.497 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   102477207, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=17.2, 数量=1, Memo=90
2025-06-21 01:50:13.709 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   102477207, 状态=Status.ALLTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=17.2, 数量=1, Memo=90
2025-06-21 01:50:13.963 | INFO | SimpleStrategy | combined: 成交信息更新 成交信息: 合约=pg2508-C-5100.DCE
编号=   100448042
方向=Direction.SHORT
开平=Offset.OPEN
价格=17.2
数量=1
成交时间=2025-06-23 21:30:08+08:00
2025-06-21 01:50:14.090 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   102486947, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.2, 数量=1, Memo=92
2025-06-21 01:50:14.222 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   102477206, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=17.6, 数量=1, Memo=90
2025-06-21 01:50:14.373 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   102477206, 状态=Status.CANCELLED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=17.6, 数量=1, Memo=90
2025-06-21 01:50:14.374 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   102486947, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.2, 数量=1, Memo=92
2025-06-21 01:50:14.437 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   102486947, 状态=Status.CANCELLED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=0.2, 数量=1, Memo=92
2025-06-21 01:50:14.452 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   103205874, 状态=Status.NOTTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=16.4, 数量=1, Memo=
2025-06-21 01:50:14.595 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=pg2508-C-5100.DCE, 编号=   103205874, 状态=Status.ALLTRADED, 方向=Direction.LONG, 开平=Offset.CLOSE, 价格=16.4, 数量=1, Memo=
2025-06-21 01:50:14.809 | INFO | SimpleStrategy | combined: 成交信息更新 成交信息: 合约=pg2508-C-5100.DCE
编号=   100544640
方向=Direction.LONG
开平=Offset.CLOSE
价格=16.4
数量=1
成交时间=2025-06-23 21:41:34+08:00
2025-06-21 01:50:14.967 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=au2507C960.SHFE, 编号=    12207677, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=0.04, 数量=1, Memo=
2025-06-21 01:50:14.997 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=au2507C952.SHFE, 编号=    12217652, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=0.04, 数量=2, Memo=
2025-06-21 01:50:15.091 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=ru2507C15000.SHFE, 编号=    12453979, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=2.0, 数量=2, Memo=
2025-06-21 01:50:15.314 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=ru2507C15250.SHFE, 编号=    12459564, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=2.0, 数量=1, Memo=
2025-06-21 01:50:15.514 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=ru2507C15500.SHFE, 编号=    12462697, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=2.0, 数量=1, Memo=
2025-06-21 01:50:15.860 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=au2507C960.SHFE, 编号=    12207677, 状态=Status.NOTTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=0.04, 数量=1, Memo=
2025-06-21 01:50:15.925 | INFO | SimpleStrategy | combined: 订单信息更新 订单信息: 合约=au2507C960.SHFE, 编号=    12207677, 状态=Status.ALLTRADED, 方向=Direction.SHORT, 开平=Offset.OPEN, 价格=0.04, 数量=1, Memo=
2025-06-21 01:50:16.161 | INFO | CTP | 行情服务器连接成功
2025-06-21 01:50:16.311 | INFO | CTP | 行情服务器登录成功
2025-06-21 01:50:20.186 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:50:20.198 | INFO | CTP | 合约信息查询成功
2025-06-21 01:50:50.629 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:53:29.715 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:55:50.240 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 01:58:38.742 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:01:39.268 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:04:59.256 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:08:35.770 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:11:44.270 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:14:34.753 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:18:00.780 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:21:19.711 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:24:49.189 | INFO | SimpleStrategy | combined: 行情数据更新
2025-06-21 02:28:27.261 | INFO | SimpleStrategy | combined: 行情数据更新
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C687908][-1621884923][ 4097]
2025-06-21 02:41:32.710 | INFO | CTP | 交易服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C6873C8][-1622015997][ 4097]
2025-06-21 02:41:32.770 | INFO | CTP | 行情服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C686FD8][-1420034042][ 4097]
2025-06-21 02:41:37.724 | INFO | CTP | 交易服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C686BE8][-1420034044][ 4097]
2025-06-21 02:41:37.784 | INFO | CTP | 行情服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C687668][-1419706361][ 4097]
2025-06-21 02:41:42.740 | INFO | CTP | 交易服务器连接断开，原因 4097
CThostFtdcUserApiImplBase::OnSessionDisconnected[000002AD3C686638][-1419706363][ 4097]
2025-06-21 02:41:42.802 | INFO | CTP | 行情服务器连接断开，原因 4097
```